## NCAA Seed Prediction — Version 1 (notebook_v1)

**Google Drive:** Upload the **whole project folder** so that under one folder you have:
- `notebooks/` (this notebook)
- `final-four-analytics-challenge-26/` (with the 3 CSVs: Training, Test, submission template)
- `Output/` (created automatically for results)

1. In the next cell: **edit `DRIVE_PROJECT_FOLDER`** to match the name of that folder as it appears in your Google Drive (e.g. `"Kaggle NCAA competition Deadline 15th March"`).
2. Local run: keep the `except` branch so `DATA_DIR` is `../final-four-analytics-challenge-26` when run from `notebooks/`.
3. Run all cells. Output is saved as `submission_v1.csv` in `Output/` (and downloaded in Colab).

In [ ]:
# Google Colab: run this first. Mount Drive, then set DATA_DIR to the folder that contains the 3 CSVs.
DRIVE_PROJECT_FOLDER = "Kaggle NCAA competition Deadline 15th March"  # EDIT: name of your uploaded folder in My Drive (must contain notebooks/ and final-four-analytics-challenge-26/)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = f"/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}/final-four-analytics-challenge-26"
except Exception:
    DATA_DIR = "../final-four-analytics-challenge-26"  # local when run from notebooks/ folder

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_FOLDS = 5

In [ ]:
import os

def _path(name_2_0, name_default):
    p2 = os.path.join(DATA_DIR, name_2_0)
    p1 = os.path.join(DATA_DIR, name_default)
    return p2 if os.path.exists(p2) else p1

train = pd.read_csv(_path("NCAA_Seed_Training_Set2.0.csv", "NCAA_Seed_Training_Set.csv"))
test = pd.read_csv(_path("NCAA_Seed_Test_Set2.0.csv", "NCAA_Seed_Test_Set.csv"))
sub = pd.read_csv(_path("submission_template2.0.csv", "submission_template.csv"))
print(train.shape, test.shape)

In [ ]:
MONTH_TO_NUM = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
                "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}

def parse_wl(val):
    if pd.isna(val) or val == "" or str(val).strip() == "0-0":
        return np.nan, np.nan, np.nan
    s = str(val).strip()
    parts = s.split("-")
    if len(parts) != 2:
        return np.nan, np.nan, np.nan
    def to_num(x):
        x = x.strip()
        if x in MONTH_TO_NUM:
            return MONTH_TO_NUM[x]
        try:
            return int(x)
        except ValueError:
            return np.nan
    w, l = to_num(parts[0]), to_num(parts[1])
    if np.isnan(w) or np.isnan(l):
        return np.nan, np.nan, np.nan
    total = w + l
    pct = w / total if total > 0 else np.nan
    return w, l, pct

def add_wl_features(df, col):
    if col not in df.columns:
        return df
    out = zip(*[parse_wl(x) for x in df[col]])
    df = df.copy()
    df[f"{col}_w"] = next(out)
    df[f"{col}_l"] = next(out)
    df[f"{col}_pct"] = next(out)
    return df

wl_cols = ["WL", "Conf.Record", "Non-ConferenceRecord", "RoadWL", "Quadrant1", "Quadrant2", "Quadrant3", "Quadrant4"]
for col in wl_cols:
    train = add_wl_features(train, col)
    test = add_wl_features(test, col)

In [ ]:
num_cols = [
    "NET Rank", "PrevNET", "AvgOppNETRank", "AvgOppNET",
    "NETSOS", "NETNonConfSOS",
    "WL_w", "WL_l", "WL_pct",
    "Conf.Record_w", "Conf.Record_l", "Conf.Record_pct",
    "Non-ConferenceRecord_w", "Non-ConferenceRecord_l", "Non-ConferenceRecord_pct",
    "RoadWL_w", "RoadWL_l", "RoadWL_pct",
    "Quadrant1_w", "Quadrant1_l", "Quadrant1_pct",
    "Quadrant2_w", "Quadrant2_l", "Quadrant2_pct",
    "Quadrant3_w", "Quadrant3_l", "Quadrant3_pct",
    "Quadrant4_w", "Quadrant4_l", "Quadrant4_pct",
]
num_cols = [c for c in num_cols if c in train.columns and c in test.columns]

for cat in ["Conference", "Bid Type"]:
    if cat not in train.columns:
        continue
    all_vals = pd.concat([train[cat], test[cat]], ignore_index=True).astype(str).fillna("__NA__")
    le = LabelEncoder()
    le.fit(all_vals.unique())
    train[f"{cat}_enc"] = le.transform(train[cat].astype(str).fillna("__NA__"))
    test[f"{cat}_enc"] = test[cat].astype(str).fillna("__NA__").map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    num_cols.append(f"{cat}_enc")

feature_cols = [c for c in num_cols if c in train.columns and c in test.columns]
print("N features:", len(feature_cols))

In [ ]:
train_seed = train.dropna(subset=["Overall Seed"]).copy()
train_seed["Overall Seed"] = train_seed["Overall Seed"].astype(int)
X = train_seed[feature_cols]
y = train_seed["Overall Seed"]
X_test = test[feature_cols]

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)

In [ ]:
ridge = Ridge(alpha=10.0, random_state=RANDOM_STATE)
cv_pred = cross_val_predict(ridge, X_imp, y, cv=KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE))
ridge_rmse = np.sqrt(np.mean((y - cv_pred) ** 2))
print("Ridge CV RMSE:", round(ridge_rmse, 4))

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
rf_pred = cross_val_predict(rf, X_imp, y, cv=KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE))
rf_rmse = np.sqrt(np.mean((y - rf_pred) ** 2))
print("RF CV RMSE:", round(rf_rmse, 4))

In [ ]:
use_rf = rf_rmse <= ridge_rmse
if use_rf:
    model = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
else:
    model = Ridge(alpha=10.0, random_state=RANDOM_STATE)
model.fit(X_imp, y)
pred = model.predict(X_test_imp)
pred = np.clip(np.round(pred), 1, 68).astype(int)
print("Model:", "RF" if use_rf else "Ridge")
print("Pred range:", pred.min(), pred.max())

In [ ]:
sub_out = sub[["RecordID"]].copy()
sub_out["Overall Seed"] = pred
os.makedirs(os.path.join(os.path.dirname(DATA_DIR), "Output"), exist_ok=True)
out_path = os.path.join(os.path.dirname(DATA_DIR), "Output", "submission_v1.csv")
sub_out.to_csv(out_path, index=False)
print("Saved:", out_path)
print(sub_out.head())

# In Google Colab: download submission_v1.csv to your computer
try:
    from google.colab import files
    files.download(out_path)
    print("Download started.")
except Exception:
    pass